# 1. Introducción a Machine Learning: Regresión Logística

En este notebook aprenderás los conceptos básicos de Machine Learning mediante un ejemplo práctico de clasificación usando regresión logística y el famoso dataset Iris.

## Objetivo
- Comprender el flujo de trabajo típico en Machine Learning.
- Aplicar regresión logística para clasificación.
- Evaluar el desempeño del modelo con métricas estándar.
- Conocer el uso de `Pipeline` de scikit-learn para organizar flujos de trabajo.

## Prerequisitos

**Conocimientos previos recomendados:**
- Python básico (variables, funciones, listas).
- Conceptos básicos de álgebra lineal y estadística.

## 1. Introducción teórica
La regresión logística es un modelo de clasificación supervisada que estima la probabilidad de que una muestra pertenezca a una clase. Es ampliamente utilizada por su simplicidad y eficiencia en problemas linealmente separables.

### ¿Por qué usar regresión logística?
- Es rápida de entrenar y fácil de interpretar.
- Permite obtener probabilidades de pertenencia a cada clase.
- Es robusta ante overfitting si se usa regularización.

**Limitaciones:**
- Solo modela relaciones lineales entre variables y la clase.
- No funciona bien si las clases no son linealmente separables.

**Parámetros importantes:**
- `penalty`: tipo de regularización ('l1', 'l2', 'elasticnet', 'none').
- `C`: inverso de la fuerza de regularización (valores bajos = más regularización).
- `solver`: algoritmo de optimización ('liblinear', 'lbfgs', 'saga', etc.).
- `max_iter`: número máximo de iteraciones para convergencia.

**Truco:**
- Si tienes muchas variables, prueba con regularización L1 para selección automática de características.

## 2. Importación de librerías

**¿Por qué estas librerías?**
- `numpy` y `pandas`: manipulación eficiente de datos.
- `matplotlib` y `seaborn`: visualización de datos y resultados.
- `sklearn`: librería estándar para machine learning en Python.

**Truco:**
- Usa `%matplotlib inline` al inicio si trabajas en Jupyter para ver los gráficos directamente en el notebook.

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 3. Carga y exploración del dataset
Usaremos el dataset Iris, incluido en scikit-learn.

**Sobre el dataset Iris:**
- 150 muestras, 4 características numéricas (largo y ancho de sépalos y pétalos).
- 3 clases: setosa, versicolor, virginica.
- Es un clásico para pruebas rápidas de algoritmos de clasificación.

**Truco:**
- Puedes cambiar el dataset por `datasets.load_wine()` o `datasets.load_breast_cancer()` para experimentar con otros problemas.

In [ ]:
iris = datasets.load_iris()
X = iris.data
y = iris.target
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y
df.head()

### Análisis exploratorio básico

**¿Qué buscar en el análisis exploratorio?**
- ¿Hay valores atípicos? ¿Las clases están balanceadas?
- ¿Qué variables parecen más relevantes para separar las clases?

**Truco:**
- Usa `sns.pairplot` para ver relaciones entre variables y clases rápidamente.

In [ ]:
df.describe()

In [ ]:
sns.pairplot(df, hue='target', palette='Set1')
plt.show()

## 4. Preprocesamiento de datos
En este caso, los datos ya están limpios y normalizados. Solo dividiremos en entrenamiento y prueba.

**¿Por qué dividir en train/test?**
- Para evaluar el modelo en datos no vistos y evitar sobreajuste.

**Parámetros importantes:**
- `test_size`: proporción de datos para prueba (0.2 = 20%).
- `random_state`: semilla para reproducibilidad.

**Truco:**
- Usa `stratify=y` para mantener la proporción de clases en ambos conjuntos si las clases están desbalanceadas.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

## 5. Construcción y entrenamiento del modelo

**Parámetros clave de LogisticRegression:**
- `penalty`: tipo de regularización ('l2' por defecto).
- `C`: controla la regularización (valores bajos = más regularización).
- `solver`: 'lbfgs' es bueno para datasets pequeños y multiclase.
- `max_iter`: aumenta si el modelo no converge.

**Truco:**
- Si ves una advertencia de convergencia, sube `max_iter` o escala los datos con `StandardScaler`.

In [ ]:
model = LogisticRegression(max_iter=200, random_state=SEED)
model.fit(X_train, y_train)

## 6. Evaluación del modelo

**¿Qué métricas mirar?**
- `accuracy_score`: proporción de aciertos.
- `classification_report`: precisión, recall y F1 por clase.
- `confusion_matrix`: errores por clase.

**Truco:**
- Si tienes clases desbalanceadas, prioriza F1-score o usa `balanced_accuracy_score`.

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión')
plt.show()

### Visualización de la frontera de decisión

Podemos visualizar cómo la regresión logística separa las clases usando solo dos características. Esto ayuda a entender el efecto de los parámetros y la linealidad del modelo.

In [ ]:
# Usamos solo dos características para visualizar en 2D
from matplotlib.colors import ListedColormap

X_vis = X[:, :2]
X_train_vis, X_test_vis, y_train_vis, y_test_vis = train_test_split(X_vis, y, test_size=0.2, random_state=SEED)
model_vis = LogisticRegression(max_iter=200, random_state=SEED)
model_vis.fit(X_train_vis, y_train_vis)

# Crear malla para graficar la frontera
gx, gy = np.meshgrid(
    np.linspace(X_vis[:,0].min()-0.5, X_vis[:,0].max()+0.5, 200),
    np.linspace(X_vis[:,1].min()-0.5, X_vis[:,1].max()+0.5, 200)
)
Z = model_vis.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

plt.figure(figsize=(8,6))
plt.contourf(gx, gy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF']))
sns.scatterplot(x=X_vis[:,0], y=X_vis[:,1], hue=y, palette='Set1', edgecolor='k')
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Frontera de decisión de la regresión logística (2D)')
plt.show()

**¿Qué observamos?**
- La frontera de decisión es lineal (recta) para cada par de clases.
- Si las clases no pueden separarse con líneas rectas, la regresión logística tendrá errores sistemáticos.
- Cambiar el parámetro `C` (regularización) puede hacer la frontera más rígida o flexible.

**Truco:**
- Prueba con diferentes pares de variables para ver cómo cambia la frontera.

### Efecto de la regularización (parámetro C)

La regularización controla la complejidad del modelo. Un valor bajo de `C` implica mayor regularización (modelo más simple), mientras que un valor alto de `C` permite que el modelo se ajuste más a los datos.

In [ ]:
# Visualización del efecto de C en la frontera de decisión
C_values = [0.01, 0.1, 1, 10, 100]
plt.figure(figsize=(15, 8))
for i, C in enumerate(C_values, 1):
    model_C = LogisticRegression(C=C, max_iter=200, random_state=SEED)
    model_C.fit(X_train_vis, y_train_vis)
    Z = model_C.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    plt.subplot(1, len(C_values), i)
    plt.contourf(gx, gy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA','#AAFFAA','#AAAAFF']))
    sns.scatterplot(x=X_vis[:,0], y=X_vis[:,1], hue=y, palette='Set1', edgecolor='k', legend=False)
    plt.title(f'C={C}')
    plt.xlabel(iris.feature_names[0])
    plt.ylabel(iris.feature_names[1])
plt.suptitle('Efecto de la regularización (C) en la frontera de decisión')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

**¿Qué se observa?**
- Con C pequeño (mucha regularización), la frontera es más rígida y puede no ajustarse bien a los datos.
- Con C grande (poca regularización), la frontera se ajusta más a los datos, pero puede sobreajustar si hay ruido.

**Truco:**
- Ajusta C usando validación cruzada (`GridSearchCV`) para encontrar el valor óptimo para tu problema.

### Comparación de métricas para diferentes valores de C

Veamos cómo cambia la precisión y el F1-score según el valor de C.

In [ ]:
from sklearn.metrics import f1_score
scores = []
for C in C_values:
    model_C = LogisticRegression(C=C, max_iter=200, random_state=SEED)
    model_C.fit(X_train_vis, y_train_vis)
    y_pred_C = model_C.predict(X_test_vis)
    acc = accuracy_score(y_test_vis, y_pred_C)
    f1 = f1_score(y_test_vis, y_pred_C, average='weighted')
    scores.append({'C': C, 'accuracy': acc, 'f1_score': f1})

scores_df = pd.DataFrame(scores)
scores_df.plot(x='C', y=['accuracy', 'f1_score'], marker='o', logx=True)
plt.title('Precisión y F1-score vs C')
plt.ylabel('Score')
plt.xlabel('C (regularización)')
plt.grid(True)
plt.show()
scores_df

**¿Qué aprendemos?**
- Hay un rango de C donde el modelo generaliza mejor.
- Si C es muy bajo o muy alto, el desempeño puede empeorar.

**Truco:**
- Siempre valida el modelo con diferentes valores de C y usa validación cruzada para evitar sobreajuste.

### Comparación de solvers y efecto del escalado de variables

El parámetro `solver` define el algoritmo de optimización. Algunos solvers requieren que los datos estén escalados para un mejor desempeño. Veamos cómo afecta esto a la precisión.

In [ ]:
from sklearn.preprocessing import StandardScaler
solvers = ['liblinear', 'lbfgs', 'saga']
results = []

# Sin escalado
for solver in solvers:
    model_s = LogisticRegression(solver=solver, max_iter=200, random_state=SEED)
    model_s.fit(X_train, y_train)
    acc = accuracy_score(y_test, model_s.predict(X_test))
    results.append({'solver': solver, 'scaled': False, 'accuracy': acc})

# Con escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
for solver in solvers:
    model_s = LogisticRegression(solver=solver, max_iter=200, random_state=SEED)
    model_s.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model_s.predict(X_test_scaled))
    results.append({'solver': solver, 'scaled': True, 'accuracy': acc})

results_df = pd.DataFrame(results)
plt.figure(figsize=(8,5))
sns.barplot(data=results_df, x='solver', y='accuracy', hue='scaled')
plt.title('Precisión según solver y escalado de variables')
plt.ylabel('Accuracy')
plt.xlabel('Solver')
plt.ylim(0.8, 1.0)
plt.show()
results_df

**¿Qué aprendemos?**
- Algunos solvers (como 'saga' y 'lbfgs') mejoran su desempeño cuando los datos están escalados.
- El escalado es especialmente importante si las variables tienen diferentes rangos.
- 'liblinear' es robusto para datasets pequeños y funciona bien sin escalado, pero para problemas grandes o con regularización L1, 'saga' es preferible.

**Truco:**
- Siempre prueba el escalado de variables y compara solvers si tu modelo no converge o la precisión es baja.

### Comparación de regularización L1 vs L2

La regularización L1 puede llevar a que algunos coeficientes sean exactamente cero (selección de variables), mientras que L2 tiende a reducir todos los coeficientes pero no los anula.

In [ ]:
# Solo algunos solvers soportan L1 (liblinear, saga)
penalties = ['l1', 'l2']
coefs = {}
for penalty in penalties:
    model_p = LogisticRegression(penalty=penalty, solver='liblinear', C=1.0, max_iter=200, random_state=SEED)
    model_p.fit(X_train_scaled, y_train)
    coefs[penalty] = model_p.coef_[0]

plt.figure(figsize=(8,5))
for penalty in penalties:
    plt.plot(coefs[penalty], marker='o', label=f'{penalty.upper()}')
plt.xticks(range(X.shape[1]), iris.feature_names, rotation=45)
plt.ylabel('Valor del coeficiente')
plt.title('Coeficientes del modelo: L1 vs L2')
plt.legend()
plt.grid(True)
plt.show()

**¿Qué observamos?**
- L1 puede anular coeficientes (llevarlos a cero), útil para selección de variables.
- L2 reduce todos los coeficientes pero no los elimina.

**Truco:**
- Si tienes muchas variables irrelevantes, prueba L1 para simplificar el modelo.

### Validación cruzada para selección de hiperparámetros

La validación cruzada permite estimar el desempeño real del modelo y elegir los mejores hiperparámetros (como C o el tipo de regularización) evitando el sobreajuste.

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'C': np.logspace(-2, 2, 10),
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}
grid = GridSearchCV(LogisticRegression(max_iter=200, random_state=SEED), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_scaled, y_train)
print('Mejores parámetros:', grid.best_params_)
print('Mejor score de validación cruzada:', grid.best_score_)


**¿Por qué usar validación cruzada?**
- Permite estimar el desempeño real del modelo en datos no vistos.
- Ayuda a elegir los mejores hiperparámetros de forma objetiva.

**Truco:**
- Usa `GridSearchCV` o `RandomizedSearchCV` para automatizar la búsqueda de hiperparámetros.

### Interpretación visual de coeficientes

Los coeficientes de la regresión logística indican la importancia y el efecto de cada variable sobre la probabilidad de pertenecer a una clase. Veamos cómo interpretarlos visualmente para las tres clases del dataset Iris.

In [ ]:
# Entrenamos modelo multiclase con regularización L2 y datos escalados
model_multi = LogisticRegression(multi_class='ovr', solver='liblinear', C=1.0, max_iter=200, random_state=SEED)
model_multi.fit(X_train_scaled, y_train)
coefs_multi = model_multi.coef_

plt.figure(figsize=(10,6))
for i, class_name in enumerate(iris.target_names):
    plt.plot(coefs_multi[i], marker='o', label=f'Clase: {class_name}')
plt.xticks(range(X.shape[1]), iris.feature_names, rotation=45)
plt.ylabel('Valor del coeficiente')
plt.title('Coeficientes por clase (One-vs-Rest)')
plt.legend()
plt.grid(True)
plt.show()

**¿Cómo interpretar?**
- Un coeficiente positivo indica que al aumentar esa variable, aumenta la probabilidad de la clase correspondiente.
- Un coeficiente negativo indica que al aumentar esa variable, disminuye la probabilidad de esa clase.
- Los coeficientes cercanos a cero indican poca influencia.

**Truco:**
- Analiza los coeficientes para entender qué variables son más relevantes para cada clase.

### Predicción probabilística y umbrales

La regresión logística permite obtener probabilidades para cada clase. El umbral por defecto es 0.5, pero puede ajustarse para priorizar precisión o recall según el problema.

In [ ]:
# Ejemplo: cambiar el umbral de decisión para la clase 0
probs = model_multi.predict_proba(X_test_scaled)
umbral = 0.7
preds_umbral = (probs[:,0] > umbral).astype(int)
# Solo para la clase 0 vs resto
y_true_bin = (y_test == 0).astype(int)
cm_umbral = confusion_matrix(y_true_bin, preds_umbral)
sns.heatmap(cm_umbral, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title(f'Matriz de confusión (umbral={umbral} para clase 0)')
plt.show()

**¿Por qué cambiar el umbral?**
- Si quieres priorizar la detección de una clase (mayor recall), baja el umbral.
- Si quieres reducir falsos positivos, sube el umbral.

**Truco:**
- Usa curvas ROC y AUC para elegir el umbral óptimo según el costo de errores en tu aplicación.

### Curvas ROC y AUC

Las curvas ROC permiten visualizar el trade-off entre sensibilidad (recall) y especificidad para diferentes umbrales. El área bajo la curva (AUC) resume la capacidad del modelo para distinguir entre clases.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarizamos las clases para ROC multiclase
n_classes = len(iris.target_names)
y_test_bin = label_binarize(y_test, classes=range(n_classes))
probs_multi = model_multi.predict_proba(X_test_scaled)

plt.figure(figsize=(8,6))
for i, class_name in enumerate(iris.target_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:,i], probs_multi[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_name} (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'k--',label='Azar')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curvas ROC por clase')
plt.legend()
plt.grid(True)
plt.show()

**¿Cómo interpretar la curva ROC?**
- Cuanto más cerca esté la curva del vértice superior izquierdo, mejor es el modelo.
- Un AUC de 1.0 es perfecto, 0.5 es azar.

**Truco:**
- Usa el AUC para comparar modelos, especialmente en problemas desbalanceados.

### Interpretación de errores

Analizar los errores del modelo ayuda a identificar patrones, clases difíciles o posibles mejoras en el preprocesamiento.

In [ ]:
# Mostramos ejemplos de errores de clasificación
# NOTA: y_pred ya fue calculado arriba con model.predict(X_test)
errors = X_test[(y_test != y_pred)]
true_labels = y_test[(y_test != y_pred)]
pred_labels = y_pred[(y_test != y_pred)]

if len(errors) > 0:
    error_df = pd.DataFrame(errors, columns=iris.feature_names)
    error_df['true_label'] = true_labels
    error_df['pred_label'] = pred_labels
    error_df['true_label_name'] = error_df['true_label'].map(lambda x: iris.target_names[x])
    error_df['pred_label_name'] = error_df['pred_label'].map(lambda x: iris.target_names[x])
    display(error_df)
else:
    print('¡No hubo errores de clasificación! El modelo acertó en todas las predicciones.')

**¿Qué hacer con los errores?**
- Analiza si hay patrones en los errores (ciertas clases, valores extremos, etc.).
- Considera agregar más datos, nuevas variables o probar modelos más complejos si los errores son sistemáticos.

**Truco:**
- Visualiza los errores para inspirar nuevas hipótesis o mejoras en el pipeline.

## 7. Buenas prácticas: Pipeline de scikit-learn

Un `Pipeline` encadena pasos de preprocesamiento y modelado en un solo objeto, reduciendo errores (como data leakage) y facilitando la reproducibilidad.

**Ventajas de usar Pipeline:**
- Evita data leakage: el escalado se ajusta **solo** con datos de entrenamiento.
- Código más limpio y fácil de mantener.
- Compatible con `GridSearchCV` y `cross_val_score`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Crear pipeline: escalado + regresión logística
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=200, random_state=SEED))
])

# Validación cruzada con pipeline
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'Accuracy promedio (CV 5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Scores por fold: {cv_scores}')

# GridSearchCV con pipeline
param_grid_pipe = {
    'clf__C': np.logspace(-2, 2, 10),
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear']
}
grid_pipe = GridSearchCV(pipe, param_grid_pipe, cv=5, scoring='accuracy')
grid_pipe.fit(X_train, y_train)
print(f'\nMejores parámetros (Pipeline): {grid_pipe.best_params_}')
print(f'Mejor accuracy (CV): {grid_pipe.best_score_:.4f}')
print(f'Accuracy en test: {grid_pipe.score(X_test, y_test):.4f}')

**¿Por qué Pipeline es una mejor práctica?**
- Garantiza que el escalado se calcula solo con los datos de entrenamiento en cada fold de validación cruzada.
- Simplifica el código de producción: un solo objeto para predecir.

**Truco:**
- En `param_grid`, usa el prefijo `nombre_paso__` (doble guión bajo) para referirte a los parámetros de cada paso del pipeline.

## 8. Discusión y Conclusiones

- La regresión logística logra una alta precisión en el dataset Iris.
- Es un buen modelo base para problemas de clasificación linealmente separables.
- El escalado de datos y la elección de solver/regularización impactan el desempeño.
- Usar `Pipeline` y `GridSearchCV` son mejores prácticas esenciales.
- Para problemas más complejos, se pueden explorar modelos no lineales o redes neuronales (ver notebooks siguientes).

## 9. Ejercicios Propuestos

Practica lo aprendido con los siguientes ejercicios:

1. **Ejercicio 1:** Cambia el dataset a `load_wine()` o `load_breast_cancer()` y repite el flujo completo. ¿Cómo cambia el accuracy? ¿Qué variables son más importantes?

2. **Ejercicio 2:** Experimenta con `stratify=y` en `train_test_split`. ¿Qué diferencia observas en la distribución de clases entre train y test?

3. **Ejercicio 3:** Usa `RandomizedSearchCV` en lugar de `GridSearchCV` para la búsqueda de hiperparámetros. ¿Es más rápido? ¿Encuentra resultados similares?

4. **Ejercicio 4 (Avanzado):** Implementa un `Pipeline` que incluya `PolynomialFeatures` antes de la regresión logística. ¿Mejora el accuracy al capturar relaciones no lineales?

## 10. Referencias y Recursos

- [Documentación de scikit-learn](https://scikit-learn.org/stable/documentation.html)
- [Regresión logística - scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Pipeline - scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [Regresión logística - Wikipedia](https://es.wikipedia.org/wiki/Regresi%C3%B3n_log%C3%ADstica)

---

📎 **Notebook siguiente:** [02. Preprocesamiento y Visualización](./02_preprocesamiento_visualizacion.ipynb)